In [1]:
from qiskit import QuantumCircuit, transpile
from qiskit_ibm_runtime import QiskitRuntimeService, SamplerV2
from qiskit.visualization import plot_histogram
import matplotlib.pyplot as plt

# ==========================================
# IBM QUANTUM CONNECTION
# ==========================================

API_KEY = "3rHiqWEZxxKhHKsxdtUF9DPokAZOctyadUnIsxI5oM-F"

CRN = "crn:v1:bluemix:public:quantum-computing:us-east:a/2f5af837dabb4d2481000377b323c1b6:691f5b63-c55d-43dd-b28c-ca3f89f398e0::"


service = QiskitRuntimeService(
    channel="ibm_quantum_platform",
    token=API_KEY,
    instance=CRN
)

print("IBM Quantum connected successfully!")

qiskit_runtime_service._discover_account:WARNING:2026-10-10 12:37:13,716: Loading account with the given token. A saved account will not be used.


IBM Quantum connected successfully!


In [2]:
backend = service.least_busy(
    operational=True,
    simulator=False,
    min_num_qubits=2
)

print("Using backend:", backend.name)

Using backend: ibm_fez


In [3]:
# ==========================================
# PART A - CORRELATED OUTCOMES
# ==========================================

qc_A = QuantumCircuit(2, 2)

qc_A.h(0)
qc_A.cx(0, 1)

qc_A.measure(0, 0)
qc_A.measure(1, 1)

print("Part A - Correlated Circuit:")
print(qc_A)

Part A - Correlated Circuit:
     ┌───┐     ┌─┐   
q_0: ┤ H ├──■──┤M├───
     └───┘┌─┴─┐└╥┘┌─┐
q_1: ─────┤ X ├─╫─┤M├
          └───┘ ║ └╥┘
c: 2/═══════════╩══╩═
                0  1 


In [4]:
# ==========================================
# PART B - OPPOSITE OUTCOMES
# ==========================================

qc_B = QuantumCircuit(2, 2)

qc_B.h(0)
qc_B.cx(0, 1)
qc_B.x(1)

qc_B.measure(0, 0)
qc_B.measure(1, 1)

print("Part B - Opposite Circuit:")
print(qc_B)

Part B - Opposite Circuit:
     ┌───┐          ┌─┐   
q_0: ┤ H ├──■───────┤M├───
     └───┘┌─┴─┐┌───┐└╥┘┌─┐
q_1: ─────┤ X ├┤ X ├─╫─┤M├
          └───┘└───┘ ║ └╥┘
c: 2/════════════════╩══╩═
                     0  1 


In [5]:
# ==========================================
# TRANSPILE FOR IBM HARDWARE
# ==========================================

qc_A_transpiled = transpile(qc_A, backend)
qc_B_transpiled = transpile(qc_B, backend)

print("Both circuits transpiled successfully!")

Both circuits transpiled successfully!


In [6]:
# ==========================================
# RUN BOTH CIRCUITS
# ==========================================

sampler = SamplerV2(mode=backend)

job = sampler.run(
    [qc_A_transpiled, qc_B_transpiled],
    shots=100
)

print("Job submitted!")
print("Job ID:", job.job_id())

C:\Users\bhavi\AppData\Local\Temp\ipykernel_17540\4124319896.py:5: DeprecationWarning: The SamplerV2 class is deprecated as of qiskit-ibm-runtime 0.50.0 and will be removed no sooner than 3 months after the release date. `qiskit_ibm_runtime.executor_sampler.Sampler` provides a new Sampler implementation that runs all the pre- and post-processing in the client side. The top-level `qiskit_ibm_runtime.Sampler` import will be updated to point to the client-side sampler in an upcoming release, with the legacy Sampler being available until the deprecation is completed.
  sampler = SamplerV2(mode=backend)


Job submitted!
Job ID: db4u8pg4qg6s73c2nhe0


In [7]:
result = job.result()

print("Job completed!")

Job completed!


In [8]:
# ==========================================
# GET RESULTS
# ==========================================

counts_A = result[0].data.c.get_counts()
counts_B = result[1].data.c.get_counts()

print("Part A Results:")
print(counts_A)

print("\nPart B Results:")
print(counts_B)

Part A Results:
{'00': 59, '11': 40, '01': 1}

Part B Results:
{'01': 44, '10': 55, '00': 1}


In [9]:
# ==========================================
# HISTOGRAMS
# ==========================================

plot_histogram(
    [counts_A, counts_B],
    legend=[
        "Part A - Correlated",
        "Part B - Opposite"
    ]
)

plt.show()